# 04a — Model Comparison

Compare multiple LLM models on the same 100-loan sample, with and without borrower description.

**Models tested:** Gemini 2.5 Flash, Gemini 2.5 Pro

**Conditions:** Structured features only vs. structured features + borrower description

## Setup

In [ ]:
import os
import pandas as pd
import numpy as np

from llm_utils import (
    load_llm_sample, run_ml_on_sample, run_llm_experiment,
    evaluate_predictions, compare_results, RESULTS_DIR
)

In [ ]:
# ── Models to compare ──────────────────────────────────────────────────────
MODELS = [
    {"api_provider": "gemini", "model_name": "gemini-2.5-flash", "label": "Gemini 2.5 Flash"},
    {"api_provider": "gemini", "model_name": "gemini-2.5-pro",   "label": "Gemini 2.5 Pro"},
    {"api_provider": "openai", "model_name": "gpt-5",            "label": "GPT-5"},
]

# Run both conditions for each model
CONDITIONS = [False, True]  # include_desc

## Load Data & Run XGBoost Baseline

In [ ]:
llm_sample = load_llm_sample()
y_true = llm_sample['loan_status'].values

xgb_probs, xgb_preds = run_ml_on_sample(llm_sample)

print(f"Sample size: {len(llm_sample)}")
print(f"Class distribution:\n{llm_sample['loan_status'].value_counts()}")
print(f"\nXGBoost predictions ready: {len(xgb_preds)} samples")

## Run All Experiments

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def run_one(model_cfg, include_desc):
    return run_llm_experiment(
        llm_sample,
        api_provider=model_cfg['api_provider'],
        model_name=model_cfg['model_name'],
        include_desc=include_desc,
        label=model_cfg['label'],
    )

# Build list of (model_cfg, include_desc) jobs
jobs = [(m, d) for m in MODELS for d in CONDITIONS]

# Run all experiments in parallel
all_results = {}
with ThreadPoolExecutor(max_workers=len(jobs)) as pool:
    futures = {pool.submit(run_one, m, d): (m['label'], d) for m, d in jobs}
    for future in futures:
        label, include_desc = futures[future]
        result = future.result()
        key = (result['label'], result['desc_tag'])
        all_results[key] = result

print(f"\nAll experiments complete: {len(all_results)} runs")

## Results Comparison

In [ ]:
# Build summary table
rows = []

# XGBoost baseline
xgb_metrics = evaluate_predictions(y_true, xgb_preds.tolist(), label="XGBoost")
rows.append({'model': 'XGBoost', 'condition': 'structured', **xgb_metrics})

# LLM results
for (label, desc_tag), result in all_results.items():
    condition = 'with_desc' if desc_tag == 'with_desc' else 'no_desc'
    rows.append({'model': label, 'condition': condition, **result['metrics']})

summary = pd.DataFrame(rows)
summary = summary.set_index(['model', 'condition'])
print(summary.to_string())

In [ ]:
import matplotlib.pyplot as plt

# Accuracy comparison chart
plot_data = summary.reset_index()
plot_data['label'] = plot_data['model'] + '\n(' + plot_data['condition'] + ')'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].barh(plot_data['label'], plot_data['accuracy'] * 100)
axes[0].set_xlabel('Accuracy (%)')
axes[0].set_title('Overall Accuracy')
axes[0].set_xlim(0, 100)
for i, v in enumerate(plot_data['accuracy']):
    axes[0].text(v * 100 + 1, i, f'{v*100:.1f}%', va='center')

# Charged Off F1
axes[1].barh(plot_data['label'], plot_data['f1_charged_off'])
axes[1].set_xlabel('F1 Score')
axes[1].set_title('Charged Off F1')
axes[1].set_xlim(0, 1)
for i, v in enumerate(plot_data['f1_charged_off']):
    axes[1].text(v + 0.02, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# Description impact: how much does desc help each model?
print("Impact of adding borrower description:")
print("=" * 60)
for model_cfg in MODELS:
    label = model_cfg['label']
    no_desc = all_results.get((label, 'no_desc'))
    with_desc = all_results.get((label, 'with_desc'))
    if no_desc and with_desc:
        acc_diff = with_desc['metrics']['accuracy'] - no_desc['metrics']['accuracy']
        f1_diff = with_desc['metrics']['f1_charged_off'] - no_desc['metrics']['f1_charged_off']
        
        # Count prediction changes
        changed = sum(a != b for a, b in zip(no_desc['predictions'], with_desc['predictions']))
        improved = sum(
            (a != t and b == t)
            for a, b, t in zip(no_desc['predictions'], with_desc['predictions'], y_true)
        )
        worsened = sum(
            (a == t and b != t)
            for a, b, t in zip(no_desc['predictions'], with_desc['predictions'], y_true)
        )
        print(f"\n{label}:")
        print(f"  Accuracy: {no_desc['metrics']['accuracy']*100:.1f}% -> {with_desc['metrics']['accuracy']*100:.1f}% ({acc_diff*100:+.1f}%)")
        print(f"  CO F1:    {no_desc['metrics']['f1_charged_off']:.3f} -> {with_desc['metrics']['f1_charged_off']:.3f} ({f1_diff:+.3f})")
        print(f"  Predictions changed: {changed}/100 (improved: {improved}, worsened: {worsened})")

## Export Results

In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)

# Save summary metrics
summary.to_csv(f"{RESULTS_DIR}/04a_model_comparison_metrics.csv")

# Save per-model detailed results
for (label, desc_tag), result in all_results.items():
    safe_name = label.lower().replace(' ', '_').replace('.', '')
    comparison = compare_results(y_true, result['predictions'], xgb_preds.tolist(), result['reasonings'])
    comparison.to_csv(f"{RESULTS_DIR}/04a_{safe_name}_{desc_tag}.csv", index=False)

print(f"Results saved to {RESULTS_DIR}/")
print(summary.to_string())